<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/Notebook_1_GEE_Forest_Area_Computation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK 1 — GEE Forest Area Computation
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [ ]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────
!pip install -q earthengine-api geemap


# ── CELL 2: Import Libraries ──────────────────────────────────────────────────
import ee
import geemap

print('✅ Libraries imported')

✅ Libraries imported


In [ ]:
# ── CELL 3: Load Datasets ─────────────────────────────────────────────────────
Hansen_GFC_2024      = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GLC_FSC30D_annual    = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/annual')
GLC_FSC30D_five_year = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/five-years-map')

print('✅ Hansen GFC 2024 loaded')
print('✅ GLC FCS30D annual loaded')
print('✅ GLC FCS30D five-year loaded')

EEException: Earth Engine client library not initialized. See http://goo.gle/ee-auth.

In [ ]:
# ── CELL 4: Select & Mask Hansen Bands ───────────────────────────────────────
treecover2000 = Hansen_GFC_2024.select('treecover2000')
datamask      = Hansen_GFC_2024.select('datamask')

treecover2000_masked = treecover2000.updateMask(treecover2000.gt(0)) \
                                    .updateMask(datamask.eq(1))

print('✅ treecover2000 masked')

# ── CELL 5: Load GLC FCS30D Year 2000 ────────────────────────────────────────
glc_2000        = GLC_FSC30D_annual.mosaic().select('b1')
glc_2000_forest = glc_2000.gte(51).And(glc_2000.lte(92)).selfMask()

print('✅ GLC FCS30D 2000 loaded')
print('✅ GLC forest mask created (classes 51-92)')

# ── CELL 6: Forest Class Definitions ─────────────────────────────────────────
forestClasses = [
    {'code': 51, 'color': '4c7300', 'name': '51 Open evergreen broadleaved'},
    {'code': 52, 'color': '006400', 'name': '52 Closed evergreen broadleaved'},
    {'code': 61, 'color': 'a8c800', 'name': '61 Open deciduous broadleaved'},
    {'code': 62, 'color': '00a000', 'name': '62 Closed deciduous broadleaved'},
    {'code': 71, 'color': '005000', 'name': '71 Open evergreen needleleaved'},
    {'code': 72, 'color': '003c00', 'name': '72 Closed evergreen needleleaved'},
    {'code': 81, 'color': '286400', 'name': '81 Open deciduous needleleaved'},
    {'code': 82, 'color': '285000', 'name': '82 Closed deciduous needleleaved'},
    {'code': 91, 'color': 'a0b432', 'name': '91 Open mixed forest'},
    {'code': 92, 'color': '788200', 'name': '92 Closed mixed forest'},
]

print(f'✅ {len(forestClasses)} forest classes defined')

✅ treecover2000 masked


In [ ]:
# ── CELL 5: FAO Regional Classification (LSIB-aligned) ───────────────────────

FAO_LSIB_REGION = {
    "Africa": {
        "Central Africa": [
            "Cameroon", "Central African Rep", "Dem Rep of the Congo",
            "Rep of the Congo", "Equatorial Guinea", "Gabon", "Sao Tome & Principe"
        ],
        "East Africa": [
            "Burundi", "Djibouti", "Eritrea", "Ethiopia", "Kenya", "Rwanda",
            "Somalia", "Sudan", "Uganda"
        ],
        "Indian Ocean Islands": [
            "Comoros", "Madagascar", "Mauritius", "Seychelles"
        ],
        "Southern Africa": [
            "Angola", "Botswana", "Lesotho", "Malawi", "Mozambique", "Namibia",
            "South Africa", "Swaziland", "Tanzania", "Zambia", "Zimbabwe"
        ],
        "West Africa": [
            "Benin", "Burkina Faso", "Cabo Verde", "Chad", "Cote d'Ivoire",
            "Gambia, The", "Ghana", "Guinea", "Guinea-Bissau", "Liberia",
            "Mali", "Mauritania", "Niger", "Nigeria", "Senegal", "Sierra Leone", "Togo"
        ],
    },
    "Americas": {
        "Caribbean": [
            "Antigua & Barbuda", "Bahamas, The", "Barbados", "Belize", "Cuba",
            "Dominica", "Dominican Republic", "Grenada", "Guyana", "Haiti",
            "Jamaica", "St Kitts & Nevis", "Saint Lucia",
            "St Vincent & the Grenadines", "Suriname", "Trinidad & Tobago"
        ],
        "Central America and Mexico": [
            "Costa Rica", "El Salvador", "Guatemala", "Honduras",
            "Mexico", "Nicaragua", "Panama"
        ],
        "North America": ["Canada", "United States"],
        "South America": [
            "Argentina", "Bolivia", "Brazil", "Chile", "Colombia",
            "Ecuador", "Paraguay", "Peru", "Uruguay", "Venezuela"
        ],
    },
    "Europe": {
        "Eastern Europe": [
            "Albania", "Armenia", "Belarus", "Bosnia & Herzegovina", "Bulgaria",
            "Croatia", "Czechia", "Estonia", "Georgia", "Hungary", "Latvia",
            "Lithuania", "Montenegro", "Poland", "Moldova", "Romania",
            "Russia", "Serbia", "Slovakia", "Slovenia", "Macedonia", "Ukraine"
        ],
        "Western Europe": [
            "Andorra", "Austria", "Belgium", "Denmark", "Finland", "France",
            "Germany", "Greece", "Iceland", "Ireland", "Italy", "Liechtenstein",
            "Luxembourg", "Monaco", "Netherlands", "Norway", "Portugal",
            "San Marino", "Spain", "Sweden", "Switzerland", "United Kingdom"
        ],
    },
    "Near East": {
        "Central Asia": [
            "Azerbaijan", "Kazakhstan", "Kyrgyzstan", "Tajikistan",
            "Turkmenistan", "Uzbekistan"
        ],
        "South/East Mediterranean": [
            "Algeria", "Cyprus", "Egypt", "Israel", "Jordan", "Lebanon",
            "Libya", "Malta", "Morocco", "Syria", "Tunisia", "West Bank"
        ],
        "West Asia": [
            "Afghanistan", "Bahrain", "Iran", "Iraq", "Kuwait", "Oman",
            "Pakistan", "Qatar", "Saudi Arabia", "Turkey",
            "United Arab Emirates", "Yemen"
        ],
    },
    "Asia and the Pacific": {
        "East Asia": ["China", "Korea, North", "Japan", "Mongolia", "Korea, South"],
        "Pacific Region": [
            "Australia", "Cook Is", "Fiji", "Kiribati", "Marshall Is",
            "Fed States of Micronesia", "Nauru", "New Zealand", "Niue",
            "Palau", "Papua New Guinea", "Samoa", "Solomon Is",
            "Tonga", "Tuvalu", "Vanuatu"
        ],
        "South Asia": ["Bangladesh", "Bhutan", "India", "Maldives", "Nepal", "Sri Lanka"],
        "Southeast Asia": [
            "Brunei", "Cambodia", "Indonesia", "Laos", "Malaysia", "Burma",
            "Philippines", "Singapore", "Thailand", "Timor-Leste", "Vietnam"
        ],
    },
}

print(f'✅ FAO_LSIB_REGION defined - {len(FAO_LSIB_REGION)} regions')

✅ FAO_LSIB_REGION defined - 5 regions


In [ ]:
#── CELL 8: Helper Functions ──────────────────────────────────────────────────
def get_all_countries(regions):
    """Return a flat sorted list of all country names."""
    countries = []
    for region in regions.values():
        for subregion_countries in region.values():
            countries.extend(subregion_countries)
    return sorted(countries)


def build_country_lookup(regions):
    """Return a dict mapping country name → {region, subregion}."""
    lookup = {}
    for region_name, subregions in regions.items():
        for subregion_name, countries in subregions.items():
            for country in countries:
                lookup[country] = {
                    "region": region_name,
                    "subregion": subregion_name
                }
    return lookup


print('✅ get_all_countries() defined')
print('✅ build_country_lookup() defined')
print(f'✅ Total countries: {len(get_all_countries(FAO_LSIB_REGION))}')

✅ get_all_countries() defined
✅ build_country_lookup() defined
✅ Total countries: 195


In [ ]:
# ── CELL 9: GEE Functions  — Countries ────────────────────────────────────────
def prepare_forest_collection(selected_regions, threshold):
    """
    Prepare a GEE FeatureCollection with forest area per country
    for a given canopy cover threshold.
    Returns a GEE object (not yet computed).
    """
    all_countries = get_all_countries(selected_regions)

    lsib_fao = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017') \
                 .filter(ee.Filter.inList('country_na', all_countries))

    forest_mask = treecover2000_masked.gte(threshold) \
                                      .selfMask() \
                                      .updateMask(datamask.eq(1))

    area_image = forest_mask.multiply(ee.Image.pixelArea().divide(1e10))

    region_areas = area_image.reduceRegions(
        collection=lsib_fao,
        reducer=ee.Reducer.sum(),
        scale=30,
    )

    region_areas = region_areas.map(lambda f: f.set('threshold', threshold))

    return region_areas


def export_forest_area(selected_regions, thresholds):
    """
    Submit one GEE export task per threshold to Google Drive.
    Returns list of tasks for monitoring.
    """
    tasks = []

    for threshold in thresholds:
        fc = prepare_forest_collection(selected_regions, threshold)

        task = ee.batch.Export.table.toDrive(
            collection=fc,
            description=f'forest_area_{threshold}',
            folder='GEE exports',
            fileNamePrefix=f'forest_area_{threshold}',
            fileFormat='CSV',
            selectors=['country_na', 'threshold', 'sum']
        )
        task.start()
        tasks.append(task)
        print(f'✅ Task submitted for threshold {threshold}%')

    return tasks


print('✅ prepare_forest_collection() defined')
print('✅ export_forest_area() defined')

✅ prepare_forest_collection() defined
✅ export_forest_area() defined


In [ ]:
# ── CELL 10: GEE Functions — US States ───────────────────────────────────────
us_state_names = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado",
    "Connecticut", "Delaware", "Florida", "Georgia", "Hawaii", "Idaho",
    "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", "Louisiana",
    "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota",
    "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada",
    "New Hampshire", "New Jersey", "New Mexico", "New York",
    "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon",
    "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
    "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington",
    "West Virginia", "Wisconsin", "Wyoming"
]

tiger_states = ee.FeatureCollection('TIGER/2018/States') \
                 .filter(ee.Filter.inList('NAME', us_state_names))


def prepare_states_forest_collection(threshold):
    """
    Prepare a GEE FeatureCollection with forest area per US state
    for a given canopy cover threshold.
    Returns a GEE object (not yet computed).
    """
    forest_mask = treecover2000_masked.gte(threshold) \
                                      .selfMask() \
                                      .updateMask(datamask.eq(1))

    area_image = forest_mask.multiply(ee.Image.pixelArea().divide(1e10))

    states_areas = area_image.reduceRegions(
        collection=tiger_states,
        reducer=ee.Reducer.sum(),
        scale=30,
        maxPixelsPerRegion=1e13
    )

    states_areas = states_areas.map(lambda f: f.set('threshold', threshold))

    return states_areas


def export_states_forest_area(thresholds):
    """
    Submit one GEE export task per threshold to Google Drive.
    Returns list of tasks for monitoring.
    """
    tasks = []

    for threshold in thresholds:
        fc = prepare_states_forest_collection(threshold)

        task = ee.batch.Export.table.toDrive(
            collection=fc,
            description=f'states_forest_area_{threshold}',
            folder='GEE exports',
            fileNamePrefix=f'states_forest_area_{threshold}',
            fileFormat='CSV',
            selectors=['NAME', 'threshold', 'sum']
        )
        task.start()
        tasks.append(task)
        print(f'✅ Task submitted for threshold {threshold}%')

    return tasks


print('✅ prepare_states_forest_collection() defined')
print('✅ export_states_forest_area() defined')
print(f'✅ {len(us_state_names)} US states defined')

✅ export_state_forest_area() defined
✅ prepare_states_forest_collection() defined
task submitted for threshold 10
task submitted for threshold 20
task submitted for threshold 30
task submitted for threshold 40
task submitted for threshold 50


In [ ]:
# ── CELL 11: Run Exports ──────────────────────────────────────────────────────
country_thresholds = [10, 20, 30, 40, 50]
state_thresholds   = [10, 20, 30, 40, 50]

print('── Submitting country tasks ──')
country_tasks = export_forest_area(FAO_LSIB_REGION, country_thresholds)

print('\n── Submitting state tasks ──')
state_tasks = export_states_forest_area(state_thresholds)

In [ ]:
#── CELL 12: Monitor Tasks ────────────────────────────────────────────────────
print('── Country tasks ──')
for task in country_tasks:
    status = task.status()
    print(f"  {status['description']}: {status['state']}")

print('\n── State tasks ──')
for task in state_tasks:
    status = task.status()
    print(f"  {status['description']}: {status['state']}")